<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/languages/python/mini_projects/experiment_bastet_terminal_game_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Bastet: A Cruel Tetris Variant

This implementation provides a text-based version of Bastet, focusing on the core game mechanics and the 'cruel' block generation. The block generation logic is simplified to consistently provide 'S' and 'Z' shaped blocks, which are often more challenging to place neatly than 'I' or 'O' blocks.

To make this game interactive, you'll need to run this code and manually step through the 'play_game' function, or implement a more sophisticated input handling system (e.g., using `curses` or a graphical library like `pygame` for a full experience).

In [19]:
import random
import time

# Game constants
BOARD_WIDTH = 20
BOARD_HEIGHT = 20
EMPTY_CELL = ' ' # Changed from '.' to space as requested
BLOCK_CHAR = '█'

# ANSI color codes for blocks
# Source: https://stackoverflow.com/questions/287871/how-to-print-colored-text-in-python
BLOCK_COLORS = {
    'I': '\033[96m', # Cyan
    'J': '\033[94m', # Blue
    'L': '\033[93m', # Yellow
    'O': '\033[97m', # White (for contrast)
    'S': '\033[92m', # Green (often S is green in Tetris)
    'T': '\033[95m', # Magenta
    'Z': '\033[91m', # Red (often Z is red in Tetris)
    'ENDC': '\033[0m' # End Color
}

# Define Tetromino shapes. Each shape is a list of 2D coordinates relative to its top-left corner.
# Each block also has different rotations.
TETROMINOES = {
    'I': [
        [[0, 0], [0, 1], [0, 2], [0, 3]], # Horizontal
        [[0, 0], [1, 0], [2, 0], [3, 0]]  # Vertical
    ],
    'J': [
        [[0, 1], [1, 1], [2, 1], [2, 0]],
        [[0, 0], [1, 0], [1, 1], [1, 2]],
        [[0, 2], [0, 1], [1, 1], [2, 1]],
        [[1, 0], [1, 1], [1, 2], [2, 2]]
    ],
    'L': [
        [[0, 0], [1, 0], [2, 0], [2, 1]],
        [[0, 2], [1, 0], [1, 1], [1, 2]],
        [[0, 1], [0, 2], [1, 2], [2, 2]],
        [[1, 0], [1, 1], [1, 2], [2, 0]]
    ],
    'O': [
        [[0, 0], [0, 1], [1, 0], [1, 1]]
    ],
    'S': [
        [[0, 1], [0, 2], [1, 0], [1, 1]],
        [[0, 0], [1, 0], [1, 1], [2, 1]]
    ],
    'T': [
        [[0, 1], [1, 0], [1, 1], [1, 2]],
        [[0, 1], [1, 0], [1, 1], [2, 1]],
        [[1, 0], [0, 1], [1, 1], [1, 2]],
        [[0, 1], [1, 1], [1, 2], [2, 1]]
    ],
    'Z': [
        [[0, 0], [0, 1], [1, 1], [1, 2]],
        [[0, 1], [1, 0], [1, 1], [2, 0]]
    ]
}

# Ensure all rotations are 4 cells (this check helps catch errors in TETROMINOES definition)
for shape_name, rotations in TETROMINOES.items():
    for i, rotation in enumerate(rotations):
        if len(rotation) != 4:
            print(f"Warning: {shape_name} rotation {i} does not have 4 cells: {rotation}")


In [20]:
def create_board():
    """Creates an empty game board."""
    return [[EMPTY_CELL for _ in range(BOARD_WIDTH)] for _ in range(BOARD_HEIGHT)]

def print_board(board, current_block=None, block_x=0, block_y=0):
    """Prints the current state of the board, optionally with a falling block."""
    display_board = [row[:] for row in board] # Make a copy to add the falling block

    if current_block is not None:
        block_shape = current_block['shape'][current_block['rotation']]
        block_color = BLOCK_COLORS.get(current_block['type'], BLOCK_COLORS['ENDC']) # Get color for current block type
        for bx, by in block_shape:
            x, y = block_x + bx, block_y + by
            if 0 <= x < BOARD_HEIGHT and 0 <= y < BOARD_WIDTH: # Only draw if within board bounds
                display_board[x][y] = block_color + BLOCK_CHAR + BLOCK_COLORS['ENDC'] # Add color codes

    # The border should match the visual width of BOARD_WIDTH characters + 2 for the side borders.
    print("\n" + "-" * (BOARD_WIDTH + 2))
    for row in display_board:
        # Ensure each cell in the row is printed correctly, accounting for color codes.
        # This joins the characters, then removes any potential extra spaces from block_char if BLOCK_CHAR was '  ' for example
        # For now, it will print BLOCK_CHAR and EMPTY_CELL normally, but the falling block will have colors.
        print("|" + "".join(row) + "|")
    print("-" * (BOARD_WIDTH + 2))

    if current_block is None:
        print("Game Over!")

def get_next_cruel_block():
    """Generates a 'cruel' block (biased towards S or Z)."""
    # For Bastet, we'll intentionally give challenging blocks.
    # A simple 'cruel' strategy is to always give S or Z blocks.
    block_type = random.choice(['S', 'Z'])
    rotation_index = random.randint(0, len(TETROMINOES[block_type]) - 1)
    return {
        'type': block_type,
        'shape': TETROMINOES[block_type],
        'rotation': rotation_index,
        'x': 0, # Starting row on the board
        'y': BOARD_WIDTH // 2 - 2 # Starting column, roughly centered
    }

In [21]:
def check_collision(board, block_shape, x, y):
    """Checks if a block collides with the board boundaries or existing blocks."""
    for bx, by in block_shape:
        new_x, new_y = x + bx, y + by
        # Check board boundaries
        if not (0 <= new_x < BOARD_HEIGHT and 0 <= new_y < BOARD_WIDTH):
            return True
        # Check collision with existing blocks
        if new_x < 0: # Allow blocks to start above the board
            continue
        if board[new_x][new_y] != EMPTY_CELL:
            return True
    return False

def place_block(board, block_shape, x, y, char=BLOCK_CHAR):
    """Places a block onto the board permanently."""
    for bx, by in block_shape:
        # Only place if within board bounds (to handle blocks starting above)
        if 0 <= (x + bx) < BOARD_HEIGHT and 0 <= (y + by) < BOARD_WIDTH:
            board[x + bx][y + by] = char

def clear_lines(board):
    """Checks for and clears full lines, returning the number of lines cleared."""
    lines_cleared = 0
    new_board = []
    for row in board:
        if EMPTY_CELL not in row:
            lines_cleared += 1
        else:
            new_board.append(row)

    # Add empty rows to the top for cleared lines
    while len(new_board) < BOARD_HEIGHT:
        new_board.insert(0, [EMPTY_CELL] * BOARD_WIDTH)

    return new_board, lines_cleared

def move_block(board, current_block, dx, dy):
    """Attempts to move the current block by (dx, dy)."""
    original_x, original_y = current_block['x'], current_block['y']
    original_rotation = current_block['rotation']

    new_x, new_y = original_x + dx, original_y + dy
    block_shape = TETROMINOES[current_block['type']][original_rotation]

    if not check_collision(board, block_shape, new_x, new_y):
        current_block['x'] = new_x
        current_block['y'] = new_y
        return True
    return False

def rotate_block(board, current_block):
    """Attempts to rotate the current block."""
    original_rotation = current_block['rotation']
    original_x, original_y = current_block['x'], current_block['y']

    next_rotation = (original_rotation + 1) % len(TETROMINOES[current_block['type']])
    new_block_shape = TETROMINOES[current_block['type']][next_rotation]

    if not check_collision(board, new_block_shape, original_x, original_y):
        current_block['rotation'] = next_rotation
        return True
    return False


In [22]:
def play_game():
    """Main game loop for Bastet with user controls and semi-continuous falling."""
    board = create_board()
    score = 0
    game_over = False
    current_block = get_next_cruel_block()

    while not game_over:
        # Current block state
        block_type = current_block['type']
        block_rotation = current_block['rotation']
        block_shape = TETROMINOES[block_type][block_rotation]
        block_x, block_y = current_block['x'], current_block['y']

        print_board(board, current_block, block_x, block_y)
        print(f"Score: {score}")
        print("Controls: 'a' (left), 'd' (right), 'w' (rotate), 's' (drop), 'q' (quit)")

        # Get user input
        command = input("Enter command (or just press Enter to let it fall): ").lower().strip()

        if command == 'q':
            game_over = True
            break
        elif command == 'a':
            move_block(board, current_block, 0, -1) # Move left
        elif command == 'd':
            move_block(board, current_block, 0, 1) # Move right
        elif command == 'w':
            rotate_block(board, current_block)
        elif command == 's':
            # Move down instantly (hard drop)
            while move_block(board, current_block, 1, 0):
                # Update block_x to reflect movement for display during drop
                block_x = current_block['x']
        elif command == '': # User pressed Enter without typing a command
            pass # Let the automatic gravity step handle the fall

        # Automatic gravity step: attempt to move block down by one
        # This happens regardless of user input (unless 's' was pressed, which already moved it down)
        if not move_block(board, current_block, 1, 0):
            # Block landed, place it permanently
            # Note: Placed blocks will not retain color as board only stores chars.
            place_block(board, block_shape, current_block['x'], current_block['y'])
            board, cleared = clear_lines(board)
            score += cleared * 100 # Simple scoring

            # Get a new cruel block
            current_block = get_next_cruel_block()
            new_block_shape = TETROMINOES[current_block['type']][current_block['rotation']]

            # Check for game over (new block spawns into collision)
            if check_collision(board, new_block_shape, current_block['x'], current_block['y']):
                game_over = True
                # Do not break immediately, allow final board print

        time.sleep(0.2) # A slight pause for visual effect between turns

    print_board(board, current_block=None) # Print final game over board without a moving block
    print(f"Final Score: {score}")
    print("GAME OVER!")


In [23]:

play_game()


----------------------
|         █          |
|        ██          |
|        █           |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
----------------------
Score: 0
Controls: 'a' (left), 'd' (right), 'w' (rotate), 's' (drop), 'q' (quit)
Enter command (or just press Enter to let it fall): d

----------------------
|                    |
|          █         |
|         ██         |
|         █          |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|                    |
|           